In [1]:
from google.colab import drive
drive.mount('/content/drive')
import zipfile
import os

# Unzip
zip_path = "/content/drive/MyDrive/NER_Amharic_Finetune/Amharic-E-commerce-Data-Extractor.zip"
extract_path = "/content/all_conll_data"

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

print(" Folder extracted.")

Mounted at /content/drive
 Folder extracted.


In [3]:
# Colab-compatible script to compare multiple NER models

from transformers import (AutoTokenizer, AutoModelForTokenClassification, Trainer,
                          TrainingArguments, DataCollatorForTokenClassification)
from datasets import load_dataset, Dataset, DatasetDict
import numpy as np
from sklearn.metrics import precision_recall_fscore_support, accuracy_score
import os, json
from google.colab import drive

# Mount Google Drive
drive.mount('/content/drive')

# Load your train and valid JSONs from Google Drive
train_file = "/content/drive/MyDrive/NER_Amharic_Finetune/train.json"
valid_file = "/content/drive/MyDrive/NER_Amharic_Finetune/valid.json"

with open(train_file, 'r', encoding='utf-8') as f:
    train_data = json.load(f)
with open(valid_file, 'r', encoding='utf-8') as f:
    valid_data = json.load(f)

dataset = DatasetDict({
    'train': Dataset.from_list(train_data),
    'validation': Dataset.from_list(valid_data)
})

# Label list extraction
def get_label_list(dataset):
    labels = set()
    for example in dataset['train']:
        labels.update(example['ner_tags'])
    return sorted(list(labels))

label_list = get_label_list(dataset)
label_to_id = {label: i for i, label in enumerate(label_list)}
id_to_label = {i: label for label, i in label_to_id.items()}

# Preprocessing function
def tokenize_and_align_labels(examples, tokenizer):
    tokenized_inputs = tokenizer(examples["tokens"], truncation=True, is_split_into_words=True, padding=True)

    labels = []
    for i, label in enumerate(examples["ner_tags"]):
        word_ids = tokenized_inputs.word_ids(batch_index=i)
        previous_word_idx = None
        label_ids = []
        for word_idx in word_ids:
            if word_idx is None:
                label_ids.append(-100)
            elif word_idx != previous_word_idx:
                label_ids.append(label_to_id[label[word_idx]])
            else:
                label_ids.append(label_to_id[label[word_idx]] if label[word_idx].startswith("I-") else -100)
            previous_word_idx = word_idx
        labels.append(label_ids)
    tokenized_inputs["labels"] = labels
    return tokenized_inputs

# Metric function
def compute_metrics(p):
    predictions, labels = p
    predictions = np.argmax(predictions, axis=2)

    # Flatten predictions and labels, filtering out -100
    true_labels = [id_to_label[l] for label in labels for l in label if l != -100]
    true_predictions = [id_to_label[p] for pred, label in zip(predictions, labels) for (p, l) in zip(pred, label) if l != -100]


    precision, recall, f1, _ = precision_recall_fscore_support(
        true_labels,
        true_predictions,
        average="macro"
    )
    acc = accuracy_score(
        true_labels,
        true_predictions
    )
    return {"accuracy": acc, "f1": f1, "precision": precision, "recall": recall}

# Main comparison function
def run_model(model_name):
    print(f"\n🚀 Training {model_name}...")
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForTokenClassification.from_pretrained(model_name, num_labels=len(label_list))

    tokenized_dataset = dataset.map(lambda x: tokenize_and_align_labels(x, tokenizer), batched=True)

    args = TrainingArguments(
        output_dir=f"/content/{model_name.replace('/', '_')}_results",
        eval_strategy="epoch",
        save_strategy="no",
        per_device_train_batch_size=8,
        per_device_eval_batch_size=8,
        num_train_epochs=3,
        weight_decay=0.01,
        logging_dir="./logs",
        logging_steps=10,
    )

    trainer = Trainer(
        model=model,
        args=args,
        train_dataset=tokenized_dataset["train"],
        eval_dataset=tokenized_dataset["validation"],
        tokenizer=tokenizer,
        data_collator=DataCollatorForTokenClassification(tokenizer),
        compute_metrics=compute_metrics,
    )

    trainer.train()
    eval_metrics = trainer.evaluate()
    print(f"✅ {model_name} evaluation: {eval_metrics}")
    return model_name, eval_metrics

# List of candidate models to try
candidate_models = [
    "xlm-roberta-base",
    "masakhane/afroxlmr-large-ner-masakhaner-1.0_2.0"
]

# Run all
results = {}
for model_name in candidate_models:
    name, metrics = run_model(model_name)
    results[name] = metrics

print("\n📊 Final Comparison:")
for name, metric in results.items():
    print(f"\nModel: {name}")
    for k, v in metric.items():
        print(f"{k}: {v:.4f}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).

🚀 Training xlm-roberta-base...


Some weights of XLMRobertaForTokenClassification were not initialized from the model checkpoint at xlm-roberta-base and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Map:   0%|          | 0/104 [00:00<?, ? examples/s]

Map:   0%|          | 0/27 [00:00<?, ? examples/s]

/tmp/ipython-input-3-2638365588.py:101: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


...

/tmp/ipython-input-3-2638365588.py:101: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Epoch,Training Loss,Validation Loss,Accuracy,F1,Precision,Recall
1,3.451400,0.746793,0.779601,0.221807,0.265596,0.208274
2,0.714700,0.407917,0.883738,0.488034,0.560730,0.475306
3,0.442900,0.289820,0.918688,0.571758,0.629460,0.550321


/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


✅ masakhane/afroxlmr-large-ner-masakhaner-1.0_2.0 evaluation: {'eval_loss': 0.28982013463974, 'eval_accuracy': 0.9186875891583453, 'eval_f1': 0.57175802079903, 'eval_precision': 0.6294600356569682, 'eval_recall': 0.550321077404154, 'eval_runtime': 1.5241, 'eval_samples_per_second': 17.715, 'eval_steps_per_second': 2.624, 'epoch': 3.0}

📊 Final Comparison:

Model: xlm-roberta-base
eval_loss: 0.3889
eval_accuracy: 0.8859
eval_f1: 0.2896
eval_precision: 0.3918
eval_recall: 0.2845
eval_runtime: 0.4701
eval_samples_per_second: 57.4350
eval_steps_per_second: 8.5090
epoch: 3.0000

Model: masakhane/afroxlmr-large-ner-masakhaner-1.0_2.0
eval_loss: 0.2898
eval_accuracy: 0.9187
eval_f1: 0.5718
eval_precision: 0.6295
eval_recall: 0.5503
eval_runtime: 1.5241
eval_samples_per_second: 17.7150
eval_steps_per_second: 2.6240
epoch: 3.0000
